# 01 — LLM Threat Model

Before writing a single line of defensive code you need a mental model of the attack surface. LLM applications are not traditional software: the "code" that determines behaviour is text — the system prompt, the user turn, retrieved documents, tool outputs — and all of it is mutable at runtime by parties you may not fully trust.

## What is an LLM attack surface?

An LLM app has at least five distinct layers where an attacker can intervene:

| Layer | What lives here | Threat |
|-------|----------------|--------|
| **Prompt** | System prompt, user input | Instruction override, role confusion |
| **Retrieval** | RAG documents, web content | Indirect injection, poisoning |
| **Tools / functions** | API calls, code exec, DB queries | Confused-deputy, over-permission |
| **Output** | Generated text, structured data | Harmful content, exfiltration |
| **Agent orchestration** | Planner, memory, sub-agents | Trust boundary violations, hijacking |

Most real incidents cross multiple layers simultaneously.

## Why — the core problem

LLMs are trained to be *helpful and instruction-following*. That property is also the vulnerability: a sufficiently convincing instruction in any input channel can override intended behaviour. Unlike traditional injection (SQL, XSS) where a parser misinterprets delimiters, LLM injection works because the model has no built-in notion of "this instruction came from the developer" vs "this instruction came from an attacker". The boundary is semantic, not syntactic.

The security community is converging on several frameworks:
- **OWASP LLM Top 10** (2023/2025 editions) — the canonical checklist
- **MITRE ATLAS** — adversarial threat landscape for AI systems
- **NIST AI RMF** — governance-level risk management

None of these replace threat modelling your specific app.

## Threat model anatomy

A practical LLM threat model has four components:
1. **Asset inventory** — what data / capabilities does the model have access to?
2. **Trust boundary map** — which inputs are from trusted sources, which from untrusted?
3. **Attack surface enumeration** — for each boundary crossing, what can an attacker inject or observe?
4. **Impact × likelihood scoring** — prioritise. Not all threats are equal.

In [ ]:
# Threat model template for LLM apps
# Fill this in for your own application — it becomes your security design doc.

THREAT_MODEL = {
    "app_name": "my-llm-app",
    "model": "claude-3-5-sonnet",
    "assets": [
        "customer PII (name, email)",
        "internal knowledge base",
        "CRM read API",
        "email send tool",
    ],
    "trust_boundaries": {
        "TRUSTED": ["system prompt", "internal tools"],
        "SEMI_TRUSTED": ["authenticated user input"],
        "UNTRUSTED": ["web search results", "email body content", "uploaded documents"],
    },
    "attack_surface": [
        {
            "layer": "prompt",
            "vector": "user turn",
            "threat": "direct prompt injection",
            "impact": "HIGH",
            "likelihood": "HIGH",
            "mitigation": "input classifier + prompt hardening",
        },
        {
            "layer": "retrieval",
            "vector": "RAG documents",
            "threat": "indirect prompt injection",
            "impact": "HIGH",
            "likelihood": "MEDIUM",
            "mitigation": "canary tokens + output monitoring",
        },
        {
            "layer": "tools",
            "vector": "email send",
            "threat": "data exfiltration via tool call",
            "impact": "CRITICAL",
            "likelihood": "LOW",
            "mitigation": "tool call approval gate + scope limits",
        },
        {
            "layer": "output",
            "vector": "rendered markdown",
            "threat": "rendered injection / XSS",
            "impact": "MEDIUM",
            "likelihood": "MEDIUM",
            "mitigation": "strip markdown in untrusted output paths",
        },
        {
            "layer": "agent",
            "vector": "sub-agent instructions",
            "threat": "trust escalation across agent boundary",
            "impact": "HIGH",
            "likelihood": "LOW",
            "mitigation": "inter-agent auth tokens + capability limits",
        },
    ],
}

def print_threat_model(tm):
    print(f"Threat Model: {tm['app_name']}  |  Model: {tm['model']}")
    print(f"\nAssets ({len(tm['assets'])}):")
    for a in tm['assets']:
        print(f"  • {a}")
    print(f"\nTrust Boundaries:")
    for level, sources in tm['trust_boundaries'].items():
        print(f"  [{level}] {', '.join(sources)}")
    print(f"\nAttack Surface ({len(tm['attack_surface'])} vectors):")
    print(f"  {'Layer':<12} {'Threat':<40} {'Impact':<10} {'Likelihood'}")
    print("  " + "-"*75)
    for v in sorted(tm['attack_surface'], key=lambda x: ['CRITICAL','HIGH','MEDIUM','LOW'].index(x['impact'])):
        print(f"  {v['layer']:<12} {v['threat']:<40} {v['impact']:<10} {v['likelihood']}")

print_threat_model(THREAT_MODEL)


## OWASP LLM Top 10 mapping

| OWASP LLM ID | Name | Covered in |
|---|---|---|
| LLM01 | Prompt Injection | NB 02–03 |
| LLM02 | Insecure Output Handling | NB 05, 11 |
| LLM03 | Training Data Poisoning | NB 10 |
| LLM04 | Model DoS | (out of scope — infra concern) |
| LLM05 | Supply Chain | (out of scope) |
| LLM06 | Sensitive Information Disclosure | NB 06, 11 |
| LLM07 | Insecure Plugin Design | NB 07 |
| LLM08 | Excessive Agency | NB 07–08 |
| LLM09 | Overreliance | NB 12 |
| LLM10 | Model Theft | (out of scope) |

This series covers LLM01, 02, 03, 06, 07, 08, 09 in depth — the ones that appear most frequently in real deployments.

In [ ]:
# Render the OWASP mapping as a quick-check checklist
OWASP_CHECKLIST = [
    ("LLM01", "Prompt Injection", "Have you tested direct and indirect injection against all input channels?"),
    ("LLM02", "Insecure Output Handling", "Is LLM output sanitised before rendering or executing?"),
    ("LLM03", "Training Data Poisoning", "Do you audit fine-tuning datasets for backdoors?"),
    ("LLM06", "Sensitive Info Disclosure", "Can the model be coerced into revealing system prompt or PII?"),
    ("LLM07", "Insecure Plugin Design", "Are tool permissions scoped to minimum necessary?"),
    ("LLM08", "Excessive Agency", "Does your agent require human approval for high-impact actions?"),
    ("LLM09", "Overreliance", "Do you validate model outputs before using them in downstream systems?"),
]

print("OWASP LLM Security Checklist")
print("="*55)
for id_, name, question in OWASP_CHECKLIST:
    print(f"\n[{id_}] {name}")
    print(f"  ? {question}")
    print(f"  □ Addressed  □ In progress  □ Not yet")
